## Notebook for Running Final Models

This notebook is designed so multiple people can run different models and split the computational workload. Each model has its own setup cell(s), formula, column list, and fit call. In most cases, you only need to run the data-preparation cells once, then run whichever model cells you are responsible for. YOU DO NOT NEED TO RUN ALL THE MODELS

#### Recommended workflow

1. Run the setup cell below. You will be asked for your name so that model results can be saved with your name attached to them.
2. Run the cell(s) that create the specific dataset(s) needed for your assigned model(s). Optionally, you can also go to the last cell and click "run all above" if your RAM supports loading all the datasets for all the models (~10 GB needed).
3. Run the model-definition and model-fitting cells for the models you were assigned. You can optionally use Parallelization as well to save time.

#### Parallelization option

There is optional parallelization code at the bottom of the notebook. This can be used to fit multiple models in parallel on the same machine.

Before using it:
- make sure all required setup cells have been run for the models you want to run. These are the cells that create the datasets.
- edit the `models_to_run` list in the parallelization cell so it contains only the model numbers you want to fit

Parallelization can save time, but it also increases memory usage because multiple model fits may run at once.

Import required dependencies and do basic feature engineering

In [1]:
# non built-in libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import geopandas as gpd
import plotly.express as px
import statsmodels.api as sm
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# built-in libraries
import os
import json
from pathlib import Path
import sys
import datetime
from multiprocessing import Pool
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# personally-defined modules
sys.path.append(os.path.join(str(Path.cwd()), "../"))  
from scripts.data_download import download_files_from_fld
from scripts.modeling.mixed_model_wrapper import (
    fit_mixed_model, 
    plot_diagnostics, 
    anova_mixed_models, 
    load_mixed_model_result,
    fit_mixed_model_parallel
)

# install reqd. datasets from Google Drive
folder_id = "1P2FRAkPrqL2nn2MNMyd4ilWbXNS_kkKD" 
data_path = os.path.join(str(Path.cwd()), "../data")
download_files_from_fld(folder_id, data_path)

# constants
START_DATE = datetime.datetime(2011, 10, 1)  # init start date of analysis to first day of FY 2012  
END_DATE = datetime.datetime(2024, 9, 30)    # init end date of analysis to last day of FY 2024

name = input("Enter your name (for labeling model results): ")
date = datetime.datetime.now().strftime("%Y-%m-%d")
r_path = r"c:\Program Files\R\R-4.5.2\bin\Rscript.exe"  # change if needed

#####################################################################################
######################### DATA PREP AND FEATURE ENGINEERING #########################
#####################################################################################

# reading in data
sent_df = pd.read_csv(os.path.join(data_path, "sentencing_data_cleaned.csv"), low_memory=False)
districts_gdf = gpd.read_file(os.path.join(data_path, "us_district_cts_bounds.geojson"))
districts_gdf = districts_gdf.rename(columns={c: c.upper() for c in districts_gdf.columns})
districts_gdf = districts_gdf[["NAME", "DISTRICT_N", "GEOMETRY"]]

# sent_df = sent_df[
#     (sent_df["DETENT_ST"] != "Other") &
#     (sent_df["DETENT_ST"].notna()) &
#     (sent_df["EDUCATION"].notna()) &
#     (sent_df["APPOINTING_PARTY"] != "None (reassignment)") & 
#     (sent_df["RACE"] != "Other") &
#     (sent_df["JUDGE_RACE"] != "Other")
# ]

sent_df = sent_df[
    (sent_df["DETENT_ST"] != "Other") &
    (sent_df["DETENT_ST"].notna()) &
    (sent_df["EDUCATION"].notna()) &
    (sent_df["APPOINTING_PARTY"] != "None (reassignment)") & 
    (sent_df["RACE"] != "Other")
]

sent_df["GOVT_SPON_DEPT"] = (
    (sent_df["5K1.1"] == 1) | 
    (sent_df["SENT_RANGE_2018_PRSNT"].isin(["Govt. Sponsored Departure", "Early Disposition/5K3.1"])) |
    (sent_df["SENT_RANGE_2004_2017"].isin(["Government Sponsored - Below Range"]))
).astype(int)

# dependent variable for one of the models (only defined for FY >= 2018)
sent_df["DOWNWARD_VAR"] = sent_df["SENT_RANGE_2018_PRSNT"].isin([
    "Govt. Sponsored Variance", "Below Range Variance"
]).astype(int)

# create guideline midpoint in years
sent_df["GL_MDPT"] = (sent_df["GL_MIN"] + sent_df["GL_MAX"]) / 24

# center PCT_ columns for easier interpretability of interactions
for col in ["PCT_MALE", "PCT_BLACK", "PCT_HISPANIC", "PCT_WHITE", "PCT_REPUBLICAN"]:
    sent_df[col] = sent_df[col] - sent_df[col].mean()

# make column called "RACE_REPRESENTATION" which if RACE=="White", then the value is centered PCT_WHITE, if
# RACE=="Black", then the value is centered PCT_BLACK, etc.
sent_df["RACE_REPRESENTATION"] = (
    sent_df["PCT_WHITE"] * (sent_df["RACE"] == "White") +
    sent_df["PCT_BLACK"] * (sent_df["RACE"] == "Black") +
    sent_df["PCT_HISPANIC"] * (sent_df["RACE"] == "Hispanic")
)

# dependent variable for if someone was sentenced above/below the guideline midpoint
sent_df["SENT_ABOVE_GL_MDPT"] = ((sent_df["MNTHS_PRSN_NO_ALT"] / 12) >= sent_df["GL_MDPT"]).astype(int)

# create LOG_SENTENCE_LENGTH_YEARS where you apply log only if MNTHS_PRSN_NO_ALT > 0, otherwise set to 0 (since log(0) is undefined)
sent_df["LOG_SENTENCE_LENGTH_YEARS"] = sent_df["MNTHS_PRSN_NO_ALT"].apply(lambda x: np.log(x / 12) if x > 0 else np.nan)

# create log of guideline midpoint
sent_df["LOG_GL_MDPT"] = np.where(sent_df["GL_MDPT"] > 0, np.log(sent_df["GL_MDPT"]), np.nan)

# treat CHC as categorical 
sent_df["CHC"] = sent_df["CHC"].astype(int).astype(str) 

# binary flags for whether the statutory minimum sentence is zero
sent_df["STAT_MIN_ZERO_FLAG"] = (sent_df["STAT_MIN"] == 0).astype(int)

# convert months of prison to years for easier model convergence
sent_df["YEARS_PRSN_NO_ALT"] = sent_df["MNTHS_PRSN_NO_ALT"] / 12

# binary flag for whether the defendant is a U.S. citizen
sent_df["CITIZEN"] = (sent_df["CITIZEN"] == "U.S. Citizen").astype(int)

# create indicator variables for Black and Hispanic defendants (with White as the reference category)
sent_df["BLACK"] = sent_df["RACE"].str.lower().eq("black").astype(int)
sent_df["HISPANIC"] = sent_df["RACE"].str.lower().eq("hispanic").astype(int)

# binary flag indicating whether the defendant commited the offense while under supervision for a prior offense
sent_df["OFF_CMTD_UNDER_SUP"] = (sent_df["OFF_CMTD_UNDER_SUP"] > 0).astype(int)

# 1 if the judge and defendant are of the same race, 0 otherwise
sent_df["JUDGE_SAME_RACE"] = (sent_df["RACE"].str.lower() == sent_df["JUDGE_RACE"].str.lower()).astype(int)

# recode education levels to reduce noise and lead to fitting less parameters
education_mapping = {
    "Less than High School": "Less than High School",
    "High School": "High School/Some College",
    "Some College": "High School/Some College",
    "Bachelor's Degree": "College Graduate",
    "Graduate Degree": "College Graduate"
}
sent_df["EDUCATION"] = sent_df["EDUCATION"].map(education_mapping)

# easier convergence
sent_df["AGE"] /= 100
sent_df["OL"] /= 43

# make reference levels for categorical variables more explicit
ref_levels = {
    "CHC": "1",
    "EDUCATION": "College Graduate",
    "DETENT_ST": "In Custody",
    "OFF_TYPE": "Drugs",
    "SEX": "Male",
    "RACE": "White",
    "JUDGE_RACE": "White",
    "JUDGE_SEX": "Male",
    "APPOINTING_PARTY": "Democratic"
}


All files already downloaded.


# Judge-grouped Models

### Model 1: Incarceration Decision by Judge with Judge Race x Offender Race Interactions

This model models incarceration decision with random slopes for offender race and offender sex. This model fits interaction terms for the race of the offender and the race of the sentencing judge.

In [ ]:
n_iter_m1 = 100_000
model_path_m1 = os.path.join("models", f"m1_incarc_AY_judge_race_inter_{n_iter_m1}_iters_{date}_{name}.json")
family_m1 = "binomial"
link_m1 = "logit"

expmnt_df_m1 = sent_df[
    (sent_df["JUDGE_NAME"].notna()) &
    (sent_df["JUDGE_SEX"].notna()) & 
    (sent_df["JUDGE_RACE"].notna()) &
    (sent_df["JUDGE_RACE"] != "Other") &
    (sent_df["RACE"] != "Other") &
    (sent_df["APPOINTING_PARTY"].notna())
].copy()

cols_to_incl_m1 = [
    "RECIEVED_PRSN_FLAG", "DIST_CRT", "RACE", "SEX", "OL", "CHC",
    "DETENT_ST", "AGE", "OFF_TYPE", "TRIAL_FLAG", "STAT_MIN_ZERO_FLAG",
    "GOVT_SPON_DEPT", "CITIZEN", "NUM_DEPENDENTS", "NUM_COUNTS", "EDUCATION",
    "JUDGE_NAME", "JUDGE_RACE", "JUDGE_SEX", "JUDGE_SAME_RACE", "APPOINTING_PARTY"
]

categorical_cols_m1 = ["JUDGE_NAME", "JUDGE_RACE", "JUDGE_SEX", "RACE", "SEX", "CHC", "OFF_TYPE", "DETENT_ST", "EDUCATION", "APPOINTING_PARTY"]

formula_m1 = "RECIEVED_PRSN_FLAG ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + STAT_MIN_ZERO_FLAG " +\
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " +\
    "+ log(NUM_COUNTS) + EDUCATION + JUDGE_RACE * RACE + JUDGE_SEX * SEX " +\
    "+ APPOINTING_PARTY * RACE + (1 + RACE || JUDGE_NAME)"

In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m1[cols_to_incl_m1],
#     formula=formula_m1,  
#     family="binomial",  
#     link="logit",
#     categorical_cols= categorical_cols_m1,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m1},  
#     optimizer = "bobyqa",  
#     n_iter = n_iter_m1,  
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m1
# )
# print(result.raw_summary)
# print(result.diagnostics["convergence_messages"])
# print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Model 2: Incarceration Decision by Judge with Indicator Variable for if Judge and Offender are Same Race

This model models incarceration decision with random slopes for offender race and offender sex. This model is simpler than Model 1 in that it does not make interaction terms between all possible judge and offender race combinations. Instead, this model uses an indicator variable for if the judge's race is the same as the offender race. This model has a lot less parameters than Model 1 and can answer if sentences are more lenient when judge and offender are the same race. This model also does not require filtering out cases where the judge race or offender race is "Other".

In [ ]:
n_iter_m2 = 100_000
model_path_m2 = os.path.join("models", f"m2_incarc_AY_judge_race_indic_{n_iter_m2}_iters_{date}_{name}.json")
family_m2 = "binomial"
link_m2 = "logit"

expmnt_df_m2 = sent_df[
    (sent_df["JUDGE_NAME"].notna()) &
    (sent_df["JUDGE_SEX"].notna()) & 
    (sent_df["JUDGE_RACE"].notna()) &
    (sent_df["APPOINTING_PARTY"].notna())
].copy()


cols_to_incl_m2 = [
    "RECIEVED_PRSN_FLAG", "DIST_CRT", "RACE", "SEX", "OL", "CHC",
    "DETENT_ST", "AGE", "OFF_TYPE", "TRIAL_FLAG", "STAT_MIN_ZERO_FLAG",
    "GOVT_SPON_DEPT", "CITIZEN", "NUM_DEPENDENTS", "NUM_COUNTS", "EDUCATION",
    "JUDGE_NAME", "JUDGE_SEX", "JUDGE_SAME_RACE", "APPOINTING_PARTY"
]

categorical_cols_m2 = ["JUDGE_NAME", "JUDGE_SEX", "RACE", "SEX", "CHC", "OFF_TYPE", "DETENT_ST", "EDUCATION", "APPOINTING_PARTY"]

formula_m2 = "RECIEVED_PRSN_FLAG ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + STAT_MIN_ZERO_FLAG " +\
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " +\
    "+ log(NUM_COUNTS) + EDUCATION + RACE + JUDGE_SAME_RACE * APPOINTING_PARTY + JUDGE_SEX * SEX " +\
    "+ (1 + RACE || JUDGE_NAME)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m2[cols_to_incl_m2],
#     formula=formula_m2,  
#     family="binomial",  
#     link="logit",
#     categorical_cols= categorical_cols_m2,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m2},  
#     optimizer = "bobyqa",  
#     n_iter = n_iter_m2,  
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m2
# )
# print(result.raw_summary)
# print(result.diagnostics["convergence_messages"])
# print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Model 3: Sentence Above/Below GL Midpoint by Judge with Interaction Terms for Judge Race and Offender Race

In [ ]:
n_iter_m3 = 100_000
model_path_m3 = os.path.join("models", f"m3_glmdpt_AY_judge_race_inter_{n_iter_m3}_iters_{date}_{name}.json")
family_m3 = "binomial"
link_m3 = "logit"

"""
Filter out cases where the GL min or max is life, since the midpoint is not well defined for those cases.
Also filter to only include cases where the sentence was actually within the guideline range, 
since we want to analyze departures in a separate model. Finally, filter to only include cases where 
we have data on education and detainment status, since those are covariates that we do not have information for for every case.
"""
expmnt_df_m3 = sent_df[
    (sent_df["GL_MIN_LIFE"] != 1) & 
    (sent_df["GL_MAX_LIFE"] != 1) & 
    (sent_df["SENT_IN_GDLNS"] == 1) &
    (sent_df["JUDGE_NAME"].notna()) &
    (sent_df["JUDGE_RACE"] != "Other") &
    (sent_df["RACE"] != "Other") & 
    (sent_df["JUDGE_SEX"].notna()) & 
    (sent_df["JUDGE_RACE"].notna()) &
    (sent_df["APPOINTING_PARTY"].notna())
].copy()

cols_to_include_m3 = [
    "SENT_ABOVE_GL_MDPT", "JUDGE_NAME", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST", 
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "JUDGE_RACE", "JUDGE_SEX", "APPOINTING_PARTY"
]

categorical_cols_m3 = [
    "JUDGE_NAME", "RACE", "EDUCATION", "OFF_TYPE", 
    "SEX", "DETENT_ST", "CHC", 'JUDGE_RACE', 'JUDGE_SEX', 'APPOINTING_PARTY'
]

formula_m3 = "SENT_ABOVE_GL_MDPT ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + log1p(NUM_DEPENDENTS)  + log(NUM_COUNTS)" +\
    "+ OFF_TYPE + TRIAL_FLAG + EDUCATION + JUDGE_RACE * RACE + JUDGE_SEX * SEX" +\
    "+ APPOINTING_PARTY * RACE + (1 + RACE || JUDGE_NAME)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m3[cols_to_include_m3],  
#     formula=formula_m3, 
#     family="binomial",  
#     link="logit",
#     categorical_cols=categorical_cols_m3,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m3},
#     optimizer = "bobyqa",  
#     n_iter=n_iter_m3,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m3
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)

### Model 4: Sentence Above/Below GL Midpoint by Judge with Indicator Variable for Judge Race == Offender Race

In [ ]:
n_iter_m4 = 100_000
model_path_m4 = os.path.join("models", f"m4_glmdpt_AY_judge_race_indic_{n_iter_m4}_iters_{date}_{name}.json")
family_m4 = "binomial"
link_m4 = "logit"

"""
Filter out cases where the GL min or max is life, since the midpoint is not well defined for those cases.
Also filter to only include cases where the sentence was actually within the guideline range, 
since we want to analyze departures in a separate model. Finally, filter to only include cases where 
we have data on education and detainment status, since those are covariates that we do not have information for for every case.
"""
expmnt_df_m4 = sent_df[
    (sent_df["GL_MIN_LIFE"] != 1) & 
    (sent_df["GL_MAX_LIFE"] != 1) & 
    (sent_df["SENT_IN_GDLNS"] == 1) &
    (sent_df["JUDGE_NAME"].notna()) &
    (sent_df["JUDGE_SEX"].notna()) & 
    (sent_df["JUDGE_RACE"].notna()) &
    (sent_df["APPOINTING_PARTY"].notna())
].copy()

cols_to_include_m4 = [
    "SENT_ABOVE_GL_MDPT", "JUDGE_NAME", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST", 
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "JUDGE_RACE", 
    "JUDGE_SEX", "JUDGE_SAME_RACE", "APPOINTING_PARTY"
]

categorical_cols_m4 = [
    "JUDGE_NAME", "RACE", "EDUCATION", "OFF_TYPE", 
    "SEX", "DETENT_ST", "CHC", 'JUDGE_SEX', 'APPOINTING_PARTY'
]

formula_m4 = "SENT_ABOVE_GL_MDPT ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + log1p(NUM_DEPENDENTS)  + log(NUM_COUNTS)" +\
    "+ OFF_TYPE + TRIAL_FLAG + EDUCATION + RACE + JUDGE_SAME_RACE * APPOINTING_PARTY + JUDGE_SEX * SEX" +\
    "+ (1 + RACE || JUDGE_NAME)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m4[cols_to_include_m4],  
#     formula=formula_m4, 
#     family="binomial",  
#     link="logit",
#     categorical_cols=categorical_cols_m4,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m4},
#     optimizer = "bobyqa",  
#     n_iter=n_iter_m4,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m4
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)

### Model 5: Sentence Length by Judge with Interaction Terms for Judge Race and Offender Race

In [ ]:
n_iter_m5 = 100_000
model_path_m5 = os.path.join("models", f"m5_log_sent_AY_judge_race_inter_{n_iter_m5}_iters_{date}_{name}.json")
family_m5 = "gaussian"
link_m5 = None

expmnt_df_m5 = sent_df[
    (sent_df["MNTHS_PRSN_NO_ALT"] > 0.03) &  # consider filtering > 1
    (sent_df["JUDGE_NAME"].notna()) &
    (sent_df["JUDGE_SEX"].notna()) & 
    (sent_df["JUDGE_RACE"].notna()) &
    (sent_df["LOG_GL_MDPT"].notna()) &
    (sent_df["APPOINTING_PARTY"].notna())
].copy()


cols_to_incl_m5 = [
    "LOG_SENTENCE_LENGTH_YEARS", "DIST_CRT", "RACE", "SEX", "OL", "CHC", 
    "DETENT_ST", "AGE", "OFF_TYPE", "TRIAL_FLAG", 
    "GOVT_SPON_DEPT", "CITIZEN", "NUM_DEPENDENTS", "NUM_COUNTS", 
    "EDUCATION", "LOG_GL_MDPT", "JUDGE_RACE", "JUDGE_SEX", "JUDGE_NAME",
    "APPOINTING_PARTY"
]

categorical_cols_m5 = [
    "JUDGE_NAME", "RACE", "SEX", "CHC", "OFF_TYPE", 
    "DETENT_ST", "EDUCATION", "JUDGE_RACE", "JUDGE_SEX", "APPOINTING_PARTY"
]

formula_m5 = "LOG_SENTENCE_LENGTH_YEARS ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + LOG_GL_MDPT " +\
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " +\
    "+ log(NUM_COUNTS) + EDUCATION + JUDGE_RACE * RACE + JUDGE_SEX * SEX " +\
    "+ APPOINTING_PARTY * RACE + (1 + RACE || JUDGE_NAME)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m5[cols_to_incl_m5], 
#     formula=formula_m5,  
#     family="gaussian",  
#     categorical_cols= categorical_cols_m5, 
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m5},  
#     optimizer = "bobyqa",  
#     n_iter = n_iter_m5,  
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m5
# )
# print(result.raw_summary)
# print(result.diagnostics["convergence_messages"])
# print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Model 6: Sentence Length by Judge with Indicator Variable for Judge Race == Offender Race

In [ ]:
n_iter_m6 = 100_000
model_path_m6 = os.path.join("models", f"m6_log_sent_AY_judge_race_indic_{n_iter_m6}_iters_{date}_{name}.json")
family_m6 = "gaussian"
link_m6 = None

expmnt_df_m6 = sent_df[
    (sent_df["MNTHS_PRSN_NO_ALT"] > 0.03) &  # consider filtering > 1
    (sent_df["JUDGE_NAME"].notna()) &
    (sent_df["JUDGE_SEX"].notna()) & 
    (sent_df["JUDGE_RACE"].notna()) &
    (sent_df["LOG_GL_MDPT"].notna()) &
    (sent_df["APPOINTING_PARTY"].notna())
].copy()

cols_to_incl_m6 = [
    "LOG_SENTENCE_LENGTH_YEARS", "DIST_CRT", "RACE", "SEX", "OL", "CHC", 
    "DETENT_ST", "AGE", "OFF_TYPE", "TRIAL_FLAG",
    "GOVT_SPON_DEPT", "CITIZEN", "NUM_DEPENDENTS", "NUM_COUNTS", 
    "EDUCATION", "LOG_GL_MDPT", "JUDGE_RACE", "JUDGE_SEX", "JUDGE_NAME",
    "JUDGE_SAME_RACE", "APPOINTING_PARTY"
]

categorical_cols_m6 = [
    "JUDGE_NAME", "RACE", "SEX", "CHC", "OFF_TYPE", 
    "DETENT_ST", "EDUCATION", "JUDGE_RACE", "JUDGE_SEX", "APPOINTING_PARTY"
]

formula_m6 = "LOG_SENTENCE_LENGTH_YEARS ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + LOG_GL_MDPT " +\
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " +\
    "+ log(NUM_COUNTS) + EDUCATION + RACE + JUDGE_SAME_RACE * APPOINTING_PARTY + JUDGE_SEX * SEX " +\
    "+ (1 + RACE || JUDGE_NAME)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m6[cols_to_incl_m6], 
#     formula=formula_m6,  
#     family="gaussian",  
#     categorical_cols= categorical_cols_m6, 
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m6},  
#     optimizer = "bobyqa",  
#     n_iter = n_iter_m6,  
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m6
# )
# print(result.raw_summary)
# print(result.diagnostics["convergence_messages"])
# print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

# District Court-grouped Models

### Model 7: Downward Variance by District Court with Interaction Terms for Court Demographics and Offender Race

Binomial model to predict for offenders who did not get a departure, if they get a downward variance or not. This model includes random slopes for race and sex for each district court. This model fits interaction terms for each combination of offender race and the pct of each demographic characteristic within a court.


In [ ]:
n_iter_m7 = 100_000
model_path_m7 = os.path.join("models", f"m7_variance_AY_dist_race_inter_{n_iter_m7}_iters_{date}_{name}.json")
family_m7 = "binomial"
link_m7 = "logit"

expmnt_df_m7 = sent_df[
    (sent_df["FISCAL_YR"] >= 2018) &  # used to be | instead of & , so this is fixed now
    (sent_df["SENT_RANGE_2018_PRSNT"].isin(
        ["Within Range", "Below Range Variance", "Govt. Sponsored Variance", "Above Range Variance"]
    ))
].copy()

cols_to_include_m7 = [
    "DOWNWARD_VAR", "DIST_CRT", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST", 
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC", "PCT_WHITE"
]

categorical_cols_m7 = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]

formula_m7 = "DOWNWARD_VAR ~ " \
    "poly(OL, 4) + CHC + TRIAL_FLAG " + \
    "+ log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) " + \
    "+ OFF_TYPE + poly(AGE, 2) + EDUCATION + CITIZEN " + \
    "+ RACE + SEX " + \
    "+ PCT_MALE * SEX " + \
    "+ PCT_BLACK * RACE + PCT_HISPANIC * RACE " + \
    "+ PCT_REPUBLICAN * RACE " + \
    "+ (1 + RACE || DIST_CRT)"

In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m7[cols_to_include_m7],
#     formula=formula_m7,
#     family="binomial",
#     link="logit",
#     categorical_cols=categorical_cols_m7,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m7},
#     optimizer = "bobyqa",
#     n_iter=n_iter_m7,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),
#     r_executable=r_path,
#     save_result_path=model_path_m7
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)


### Model 8: Downward Variance by District Court with Race Representation Variable

In [ ]:
n_iter_m8 = 100_000
model_path_m8 = os.path.join("models", f"m8_variance_AY_dist_race_rep_{n_iter_m8}_iters_{date}_{name}.json")
family_m8 = "binomial"
link_m8 = "logit"

expmnt_df_m8 = sent_df[
    (sent_df["FISCAL_YR"] >= 2018) &  # used to be | instead of & , so this is fixed now
    (sent_df["SENT_RANGE_2018_PRSNT"].isin(
        ["Within Range", "Below Range Variance", "Govt. Sponsored Variance", "Above Range Variance"]
    ))
].copy()

cols_to_include_m8 = [
    "DOWNWARD_VAR", "DIST_CRT", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST", 
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC", "PCT_WHITE", "RACE_REPRESENTATION"
]

categorical_cols_m8 = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]

formula_m8 = "DOWNWARD_VAR ~ " \
    "poly(OL, 4) + CHC + TRIAL_FLAG " + \
    "+ log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) " + \
    "+ OFF_TYPE + poly(AGE, 2) + EDUCATION + CITIZEN " + \
    "+ RACE + SEX + RACE_REPRESENTATION " + \
    "+ PCT_MALE * SEX + PCT_REPUBLICAN * RACE " + \
    "+ (1 + RACE || DIST_CRT)"

In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m8[cols_to_include_m8],
#     formula=formula_m8,
#     family="binomial",
#     link="logit",
#     categorical_cols=categorical_cols_m8,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m8},
#     optimizer = "bobyqa",
#     n_iter=n_iter_m8,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),
#     r_executable=r_path,
#     save_result_path=model_path_m8
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)


### Model 9: Above/Below Guideline Midpoint by District Court with Court Composiiton x Offender Race Interactions

In [ ]:
n_iter_m9 = 100_000
model_path_m9 = os.path.join("models", f"m9_glmdpt_AY_dist_race_inter_{n_iter_m9}_iters_{date}_{name}.json")
family_m9 = "binomial"
link_m9 = "logit"

"""
Filter out cases where the GL min or max is life, since the midpoint is not well defined for those cases.
Also filter to only include cases where the sentence was actually within the guideline range,
since we want to analyze departures in a separate model. Finally, filter to only include cases where
we have data on education and detainment status, since those are covariates that we do not have information for for every case.
"""
expmnt_df_m9 = sent_df[
    (sent_df["GL_MIN_LIFE"] != 1) &
    (sent_df["GL_MAX_LIFE"] != 1) &
    (sent_df["SENT_IN_GDLNS"] == 1)
].copy()

cols_to_include_m9 = [
    "SENT_ABOVE_GL_MDPT", "DIST_CRT", "RACE",
    "EDUCATION", "OFF_TYPE", "SEX", "AGE",
    "ANY_CRIM_HIST_FLAG", "DETENT_ST",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS",
    "NUM_DEPENDENTS", "CITIZEN", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC"
]

categorical_cols_m9 = ["DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"]

formula_m9 = "SENT_ABOVE_GL_MDPT ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + " + \
    "log1p(NUM_DEPENDENTS) + log(NUM_COUNTS) + " + \
    "OFF_TYPE + TRIAL_FLAG + EDUCATION + CITIZEN + " + \
    "RACE + SEX + " + \
    "PCT_MALE * SEX + " + \
    "PCT_BLACK * RACE + PCT_HISPANIC * RACE + " + \
    "PCT_REPUBLICAN * RACE + " + \
    "(1 + RACE || DIST_CRT)"

In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m9[cols_to_include_m9],  
#     formula=formula_m9,
#     family="binomial",  
#     link="logit",
#     categorical_cols=categorical_cols_m9,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m9},
#     optimizer = "bobyqa",  
#     n_iter=n_iter_m9,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m9
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)

### Model 10: Above/Below Guideline Midpoint by District Court with Race Representation Variable

In [ ]:
n_iter_m10 = 100_000
model_path_m10 = os.path.join("models", f"m10_glmdpt_AY_dist_race_rep_{n_iter_m10}_iters_{date}_{name}.json")
family_m10 = "binomial"
link_m10 = "logit"

"""
Filter out cases where the GL min or max is life, since the midpoint is not well defined for those cases.
Also filter to only include cases where the sentence was actually within the guideline range,
since we want to analyze departures in a separate model. Finally, filter to only include cases where
we have data on education and detainment status, since those are covariates that we do not have information for for every case.
"""
expmnt_df_m10 = sent_df[
    (sent_df["GL_MIN_LIFE"] != 1) &
    (sent_df["GL_MAX_LIFE"] != 1) &
    (sent_df["SENT_IN_GDLNS"] == 1)
].copy()

cols_to_include_m10 = [
    "SENT_ABOVE_GL_MDPT", "DIST_CRT", "RACE",
    "EDUCATION", "OFF_TYPE", "SEX", "AGE",
    "ANY_CRIM_HIST_FLAG", "DETENT_ST",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS",
    "NUM_DEPENDENTS", "CITIZEN", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC", "RACE_REPRESENTATION"
]

categorical_cols_m10 = ["DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"]

formula_m10 = "SENT_ABOVE_GL_MDPT ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + " + \
    "log1p(NUM_DEPENDENTS) + log(NUM_COUNTS) + " + \
    "OFF_TYPE + TRIAL_FLAG + EDUCATION + " + \
    "RACE + SEX + RACE_REPRESENTATION + " + \
    "PCT_MALE * SEX + PCT_REPUBLICAN * RACE + " + \
    "(1 + RACE || DIST_CRT)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m10[cols_to_include_m10],  
#     formula=formula_m10,
#     family="binomial",  
#     link="logit",
#     categorical_cols=categorical_cols_m10,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m10},
#     optimizer = "bobyqa",  
#     n_iter=n_iter_m10,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),  
#     r_executable=r_path,
#     save_result_path=model_path_m10
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)

### Model 11: Incarceration Decision by District Court with District Court Race Interaction with Offender Race

In [ ]:
n_iter_m11 = 100_000
model_path_m11 = os.path.join("models", f"m11_incarc_AY_dist_race_inter_{n_iter_m11}_iters_{date}_{name}.json")
family_m11 = "binomial"
link_m11 = "logit"

expmnt_df_m11 = sent_df.copy()

cols_to_include_m11 = [
    "RECIEVED_PRSN_FLAG", "DIST_CRT", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST", "STAT_MIN_ZERO_FLAG",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "GOVT_SPON_DEPT", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC"
]

categorical_cols_m11 = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]

formula_m11 = "RECIEVED_PRSN_FLAG ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + STAT_MIN_ZERO_FLAG " + \
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " + \
    "+ log(NUM_COUNTS) + EDUCATION + RACE + SEX " + \
    "+ PCT_MALE * SEX " + \
    "+ PCT_BLACK * RACE + PCT_HISPANIC * RACE " + \
    "+ PCT_REPUBLICAN * RACE " + \
    "+ (1 + RACE || DIST_CRT)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m11[cols_to_include_m11],
#     formula=formula_m11,
#     family="binomial",
#     link="logit",
#     categorical_cols=categorical_cols_m11,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m11},
#     optimizer = "bobyqa",
#     n_iter=n_iter_m11,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),
#     r_executable=r_path,
#     save_result_path=model_path_m11
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)


### Model 12: Incarceration Decision by District Court with Race Representation Variable

In [ ]:
n_iter_m12 = 100_000
model_path_m12 = os.path.join("models", f"m12_incarc_AY_dist_race_rep_{n_iter_m12}_iters_{date}_{name}.json")
family_m12 = "binomial"
link_m12 = "logit"

expmnt_df_m12 = sent_df.copy()

cols_to_include_m12 = [
    "RECIEVED_PRSN_FLAG", "DIST_CRT", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST", "STAT_MIN_ZERO_FLAG",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC", "RACE_REPRESENTATION"
]

categorical_cols_m12 = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]

formula_m12 = "RECIEVED_PRSN_FLAG ~ " + \
    "poly(OL, 4) + CHC + DETENT_ST + poly(AGE, 2) + STAT_MIN_ZERO_FLAG " + \
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " + \
    "+ log(NUM_COUNTS) + EDUCATION + RACE + SEX + RACE_REPRESENTATION " + \
    "+ PCT_MALE * SEX " + \
    "+ PCT_REPUBLICAN * RACE " + \
    "+ (1 + RACE || DIST_CRT)"

In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m12[cols_to_include_m12],
#     formula=formula_m12,
#     family="binomial",
#     link="logit",
#     categorical_cols=categorical_cols_m12,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m12},
#     optimizer = "bobyqa",
#     n_iter=n_iter_m12,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),
#     r_executable=r_path,
#     save_result_path=model_path_m12
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)

### Model 13: Log(Sentence Length) by District Court with District Level Race Interactions

In [ ]:
n_iter_m13 = 100_000
model_path_m13 = os.path.join("models", f"m13_log_sent_AY_dist_race_inter_{n_iter_m13}_iter_{date}_{name}.json")
family_m13 = "gaussian"
link_m13 = None

expmnt_df_m13 = sent_df.copy()

cols_to_include_m13 = [
    "LOG_SENTENCE_LENGTH_YEARS", "DIST_CRT", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST", "STAT_MIN_ZERO_FLAG",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "GOVT_SPON_DEPT", "LOG_GL_MDPT", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC", "RACE_REPRESENTATION"
]

categorical_cols_m13 = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]

formula_m13 = "LOG_SENTENCE_LENGTH_YEARS ~ " + \
    "poly(OL, 4) + CHC + LOG_GL_MDPT + DETENT_ST + poly(AGE, 2) " + \
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " + \
    "+ log(NUM_COUNTS) + EDUCATION + RACE + SEX " + \
    "+ PCT_MALE * SEX " + \
    "+ PCT_BLACK * RACE + PCT_HISPANIC * RACE " + \
    "+ PCT_REPUBLICAN * RACE " + \
    "+ (1 + RACE || DIST_CRT)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m13[cols_to_include_m13],
#     formula=formula_m13,
#     family="gaussian",
#     categorical_cols=categorical_cols_m13,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m13},
#     optimizer = "bobyqa",
#     n_iter=n_iter_m13,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),
#     r_executable=r_path,
#     save_result_path=model_path_m13
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)


### Model 14: Log(Sentence Length) by District Court with Race Representation Variable

In [ ]:
n_iter_m14 = 100_000
model_path_m14 = os.path.join("models", f"m14_log_sent_AY_dist_race_rep_{n_iter_m14}_iters_{date}_{name}.json")
family_m14 = "gaussian"
link_m14 = None

expmnt_df_m14 = sent_df.copy()

cols_to_include_m14 = [
    "LOG_SENTENCE_LENGTH_YEARS", "DIST_CRT", "RACE", 
    "EDUCATION", "OFF_TYPE", "SEX", "AGE", 
    "ANY_CRIM_HIST_FLAG", "DETENT_ST",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS", 
    "NUM_DEPENDENTS", "CITIZEN", "GOVT_SPON_DEPT", "LOG_GL_MDPT", "PCT_REPUBLICAN",
    "PCT_MALE", "PCT_BLACK", "PCT_HISPANIC", "RACE_REPRESENTATION"
]

categorical_cols_m14 = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]

formula_m14 = "LOG_SENTENCE_LENGTH_YEARS ~ " + \
    "poly(OL, 4) + CHC + LOG_GL_MDPT + DETENT_ST + poly(AGE, 2) " + \
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " + \
    "+ log(NUM_COUNTS) + EDUCATION + RACE + SEX + RACE_REPRESENTATION " + \
    "+ PCT_MALE * SEX " + \
    "+ PCT_REPUBLICAN * RACE " + \
    "+ (1 + RACE || DIST_CRT)"


In [ ]:
# result = fit_mixed_model(
#     expmnt_df_m14[cols_to_include_m14],
#     formula=formula_m14,
#     family="gaussian",
#     categorical_cols=categorical_cols_m14,
#     reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols_m14},
#     optimizer = "bobyqa",
#     n_iter=n_iter_m14,
#     r_script_path=os.path.join("r", "fit_mixed_model.R"),
#     r_executable=r_path,
#     save_result_path=model_path_m14
# )
# print(result.diagnostics["convergence_messages"])
# print(result.raw_summary)

## Miscellaneous Models

## Run Parallelized Models

In [ ]:
models_to_run = [] # enter integers corresponding to which models you want to run, e.g. [1, 3, 5] to run models 1, 3, and 5 from above

# delete all expmnt_df_mX not in models_to_run to save memory
dfs = {k: v for k, v in globals().items() if k.startswith("expmnt_df_m")}
for k in list(dfs.keys()):
    model_num = int(k.split("_m")[-1])
    if model_num not in models_to_run:
        del globals()[k]

# create parameter dictionaries for each model
jobs = [
    {
        "data": globals()[f"expmnt_df_m{i}"],
        "formula": globals()[f"formula_m{i}"],
        "family": globals()[f"family_m{i}"],
        "link": globals()[f"link_m{i}"],
        "categorical_cols": globals()[f"categorical_cols_m{i}"],
        "reference_levels": {k: v for k, v in ref_levels.items() if k in globals()[f"categorical_cols_m{i}"]},
        "optimizer": "bobyqa",
        "n_iter": globals()[f"n_iter_m{i}"],
        "r_script_path": os.path.join("r", "fit_mixed_model.R"),  
        "r_executable": r_path,
        "save_result_path": globals()[f"model_path_m{i}"],
    }
    for i in models_to_run
]

if not jobs:
    print("No models selected. Populate models_to_run with model numbers before running this cell.")
elif __name__ == "__main__":
    with Pool(processes=len(jobs)) as pool:
        results = pool.map(fit_mixed_model_parallel, jobs)
